<a href="https://colab.research.google.com/github/blankperson-cyber/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task Type:** Regression / Scoring (Continuous Value Prediction)

**Why:**
We are predicting a continuous score representing the Quality of Experience (QoE) or Estimated Session Retention Probability based on multidimensional performance metrics (e.g., render latency, frame drops, network bandwidth, memory footprint, and asset complexity). Rather than just classifying a session as "good" or "bad" (which hides the subtle degradations that trigger a user to bounce), scoring allows us to model a continuous degradation curve. This score acts as a dynamic utility function to determine exactly when to degrade asset quality (e.g., streaming a lower-resolution Gaussian Splat point-cloud) before the user actually experiences a jarring stutter.

In [16]:
# Verify task target type (scoring a continuous variable)
import pandas as pd
import numpy as np

# Mocking the output shape of our scoring task
mock_qoe_scores = np.random.uniform(0.0, 1.0, size=5)
print("Example predicted QoE scores (0 = worst, 1 = flawless):")
print(mock_qoe_scores)

Example predicted QoE scores (0 = worst, 1 = flawless):
[0.10085969 0.78661373 0.87657425 0.56647549 0.96448838]


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**What we are predicting (Target):** A Perceptual QoE Score ($Q$), scaled from $0.0$ (immediate session abandonment due to unplayability) to $1.0$ (perfectly smooth, high-fidelity rendering).

**Label Source:** This target is an observed outcome proxy. While true "perception" is subjective, we construct our proxy using concrete, observed telemetry: User Engagement Duration relative to the average page session, combined with Session-End State (e.g., did they close the tab during a sustained frame-rate drop?).

* **Formula Proxy:** $Q = f(\text{Frame Rate Ratio}) \times g(\text{Asset Load Time}) \times I(\text{Active Session Completion})$

* It is not a human-labeled metric or a rigid rule, but an observed behavioral proxy harvested directly from real client telemetry.

In [17]:
# Illustrating how the target proxy column 'qoe_score' is engineered
# from raw telemetry measurements.
df_target_sketch = pd.DataFrame({
    'session_id': ['s101', 's102', 's103'],
    'frame_rate_drop_pct': [0.05, 0.45, 0.12], # proxy for stuttering
    'load_time_seconds': [1.2, 8.5, 3.1],       # proxy for payload friction
    'user_bounced_early': [0, 1, 0]             # behavioral indicator
})

# A simple target generation formula mimicking our data engineering pipeline
df_target_sketch['qoe_score'] = (
    (1 - df_target_sketch['frame_rate_drop_pct']) *
    (1 / (1 + df_target_sketch['load_time_seconds'] * 0.1)) *
    (1 - 0.5 * df_target_sketch['user_bounced_early'])
).clip(0.0, 1.0)

print(df_target_sketch[['session_id', 'qoe_score']])

  session_id  qoe_score
0       s101   0.848214
1       s102   0.148649
2       s103   0.671756


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Success Metric:** Root Mean Squared Error (RMSE) on the test partition, paired with Mean Absolute Error (MAE) for interpretability.What 'Good' Looks Like:Because our target $Q \in [0, 1]$, an MAE $< 0.08$ means our model can predict the user's perceptual quality degradation to within 8% of their actual threshold.In production, the ultimate business success metric supported by this model is a 15% reduction in asset-induced session bounce rates without lowering the baseline visual fidelity for capable high-end devices.

In [18]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Dummy evaluation run to prove our success metric framework is ready
y_true = np.array([0.90, 0.35, 0.78, 0.85, 0.20])
y_pred = np.array([0.86, 0.41, 0.72, 0.88, 0.28]) # Model predictions

mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))

print(f"Target MAE: < 0.08 | Validation MAE: {mae:.4f} (PASS)" if mae < 0.08 else f"Validation MAE: {mae:.4f} (FAIL)")
print(f"Validation RMSE: {rmse:.4f}")

Target MAE: < 0.08 | Validation MAE: 0.0540 (PASS)
Validation RMSE: 0.0567


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**Unit of Analysis: One row = One unique user media session.**

Each row represents a single session where a user interacted with a specific immersive asset. The features capture the client device capabilities, network conditions, asset complexity, and our target variable.

In [19]:
# Creating the real schema layout for your slice of the database
import pandas as pd

# Representing exactly how one row (one session) looks
session_data = {
    "session_id": ["sess_90823", "sess_90824"],
    "device_gpu_tier": [3, 1],                    # Client hardware capability (1-3)
    "network_rtt_ms": [42, 180],                  # Round Trip Time (network delay)
    "asset_poly_count": [450000, 120000],          # Asset weight / 3D complexity
    "render_pipeline": ["WebGL2", "WebGPU"],      # Software engine
    "ram_utilization_mb": [1420, 890],            # Memory footprint
    "qoe_score": [0.92, 0.41]                     # The target variable to predict
}

df_session = pd.DataFrame(session_data)
print("DataFrame Shape (Unit of Analysis: 1 Row = 1 User Media Session):")
print(df_session.shape)
print("\nDataFrame Sample:")
df_session.head()

DataFrame Shape (Unit of Analysis: 1 Row = 1 User Media Session):
(2, 7)

DataFrame Sample:


,session_id,device_gpu_tier,network_rtt_ms,asset_poly_count,render_pipeline,ram_utilization_mb,qoe_score
0,sess_90823,3,42,450000,WebGL2,1420,0.92
1,sess_90824,1,180,120000,WebGPU,890,0.41


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A hardcoded rule-set (e.g., if ram > 1GB and rtt > 150ms: drop_quality()) fails to capture the intricate, non-linear relationships of modern web engines.

1. Cooperative Device Capabilities: A device with a top-tier GPU can handle a massive 3D Gaussian Splat payload even on a high-latency connection once loaded, whereas a low-end mobile device might stutter instantly despite having fiber-optic speeds.

2. Complex Feature Interactions: The threshold of performance degradation is a moving target. System resources, background processes, browser engines, and asset optimization techniques (like level-of-detail mesh streaming) interact in highly complex, high-dimensional spaces.

3. Adaptability: An ML model can predict a continuous curve of expected QoE, enabling our content delivery network (CDN) to stream fluid, micro-adjusted asset states dynamically, rather than relying on clunky, binary high/low quality switches that disrupt user immersion.

In [20]:
# A simple visualization demonstrating where a fixed rule breaks
# Lower GPU tiers might handle light assets, but crash on heavy ones—interaction is non-linear.
def fixed_rule_engine(gpu, asset_weight):
    if asset_weight > 300000 and gpu < 2:
        return "Drop Quality"
    return "Keep High Quality"

# In reality, a low-end GPU (tier 1) might still handle a 350k asset fine if the RAM utilization is extremely low,
# or fail on a 100k asset if network jitter is high. Fixed rules cannot dynamically trade features off.
test_cases = [
    {"gpu": 1, "asset_weight": 350000, "note": "Classified as 'Drop' by rules, but could be smooth under low latency"},
    {"gpu": 2, "asset_weight": 250000, "note": "Classified as 'Keep' by rules, but will stutter under high background RAM load"}
]

for tc in test_cases:
    action = fixed_rule_engine(tc["gpu"], tc["asset_weight"])
    print(f"Rule output: {action:18} | Context: {tc['note']}")

Rule output: Drop Quality       | Context: Classified as 'Drop' by rules, but could be smooth under low latency
Rule output: Keep High Quality  | Context: Classified as 'Keep' by rules, but will stutter under high background RAM load


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.